# MPPP — 03 COLMAP alignment (Navcam, optionally + Mastcam-Z 34 mm) (MPPP 0.40.0)

PDS products in, a prior-aligned COLMAP block out: images processed and masked, features matched, a weighted bundle adjustment from the rover poses (stations with fewer than `LOCALIZE_MIN_IMAGES` images get no waypoint position prior), a health report and the `error_input/` folder for notebooks 04 (camera models) and 05 (error analysis). Every setting is in the `parameters` cell (section 0); `scripts/run_scapes.py` runs this notebook for several sites by overriding them.

**Sites** (`SITES`: name → sol range; add your own):

| site | sols | site | sols |
|---|---|---|---|
| butler_landing | 1–14 | airey_hill | 960–991 |
| van_zyl | 49–71 | bunsen_peak | 1066–1095 |
| rochette | 178–190 | overlook_mountain | 1150–1155 |
| seitah_north | 238–279 | pearce_canyon | 1183–1218 |
| sid | 361–378 | pico_turquino | 1307–1310 |
| rose_river_falls | 448–450 | rio_chiquito | 1333–1337 |
| rockytop (difficult: winter sun) | 461–530 | south_arm | 1408–1412 |
| enchanted_lake | 556–590 | bell_island (cross-station packing) | 1451–1467 |
| whale_mountain | 606–610 | taylorfjellet_large (widest LMST range) | 1601–1645 |
| threeforks_south | 652–674 | taylorfjellet | 1606–1625 |
| threeforks (easy: flat ground) | 652–692 | origny_large | 1765–1813 |
| threeforks_north | 680–692 | origny | 1781–1813 |
| knob_mountain | 698–706 | olifants | 1880–1889 |
| berea | 732–738 | groloy | 1922–1934 |
| belva_crater | 784–815 | hippo_pools | 1947–1955 |
| tuxedo_park | 897–908 | marble_mountain | 1965–1979 |

`ADD_NEARBY_WAYPOINTS` adds every other visit within that many metres of a station of the sol range (0 = off).

**What the run does:** select one product per exposure → process (8-bit PNG, terrain mask, CAHV + waypoint pose; images pointing above `SKY_ELEVATION_DEG`, taken outside `LMST_WINDOW_H`, or with more than `MAX_SATURATED_FRACTION` saturated pixels are left out) → one COLMAP camera per eye (rational Navcam lens, consensus start cameras and rig; the stereo baseline is held and sets the scale) → SIFT at native resolution inside the masks → exhaustive matching → triangulation from the rover poses and a weighted bundle adjustment with position and attitude priors (frames that do not align with the majority are excluded; with Mastcam-Z, the Navcam block is solved first and held) → health report with a top-down station map → export.

**Frame.** The block lives in East–North–Up metres of the landing site frame minus a rounded offset (`proj.offset`, recorded in `project.json` and `error_input/summary.json`). The attitude priors (`ATTITUDE_PRIOR_DEG`) keep the cardinal directions; the health check `block_rotation_deg` reports any common rotation of the block away from them.

**COLMAP GUI.** `open_in_colmap.bat` in the project folder starts COLMAP (`COLMAP_BAT`) with the database, the images and the refined model loaded.

This kernel needs pip `pycolmap` 4.2 and `pyceres` 2.6 (`pip install -e .[sfm]`); GPU extraction and matching run in `GPU_PY`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda: duplicate Intel OpenMP guard; before torch
import sys, json
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import mppp
import pycolmap, pyceres
from mppp.select import find_imgs
from mppp.sfm import (SfmProject, select_best_products, extract_features, build_database, prior_overlap_pairs,
                      match, reconstruct, export_for_error, check_gpu_python, check_ba_environment)
from mppp.sfm.health import assess_alignment, health_table, write_health
from mppp.sfm.reconstruction import convergence_statistics
print("MPPP", mppp.__version__, "from", Path(mppp.__file__).parent, "| pycolmap", pycolmap.__version__,
      "(CUDA:", pycolmap.has_cuda, ") | pyceres", pyceres.__version__)
check_ba_environment()        # stops here if this kernel cannot run the weighted BA (e.g. it is the conda CUDA env)

## 0. Settings
Every setting of the run is in the cell below. `scripts/run_scapes.py` overrides them, one site at a time.

In [ ]:
# --- data and output ------------------------------------------------------------------------------------
PDS_DIR     = Path("D:/data/m2020")          # the local PDS archive (searched recursively)
SCAPES_ROOT = Path("D:/scapes/colmap")       # one WORK folder per site below this
# Optional GPU for feature extraction and matching (Windows): the python.exe of a second environment with the
# conda-forge CUDA pycolmap of the same COLMAP version (README, "GPU on Windows").  None = this kernel, CPU.
GPU_PY = r"C:\Users\ctate\AppData\Local\miniconda3\envs\mppp_gpu\python.exe"
COLMAP_BAT = r"D:\tools\colmap-x64-windows-nocuda\COLMAP.bat"   # for open_in_colmap.bat (None: search the usual places)
WORKERS = 4                                  # image-processing worker processes

# --- site ---------------------------------------------------------------------------------------------------
SITES = {                                    # sol range of the Navcam (+ Mastcam-Z) block; add your own
    "butler_landing":       (   1,   14),
    "van_zyl":              (  49,   71),
    "rochette":             ( 178,  190),
    "seitah_north":         ( 238,  279),
    "sid":                  ( 361,  378),
    "rose_river_falls":     ( 448,  450),
    "rockytop":             ( 461,  530),
    "enchanted_lake":       ( 556,  590),
    "whale_mountain":       ( 606,  610),
    "threeforks_south":     ( 652,  674),
    "threeforks_north":     ( 680,  692),
    "threeforks":           ( 652,  692),
    "knob_mountain":        ( 698,  706),
    "berea":                ( 732,  738),
    "belva_crater":         ( 784,  815),
    "tuxedo_park":          ( 897,  908),
    "airey_hill":           ( 960,  991),
    "bunsen_peak":          (1066, 1095),
    "overlook_mountain":    (1150, 1155),
    "pearce_canyon":        (1183, 1218),
    "pico_turquino":        (1307, 1310),
    "rio_chiquito":         (1333, 1337),
    "south_arm":            (1408, 1412),
    "bell_island":          (1451, 1467),
    "taylorfjellet_large":(1601, 1645),
    "taylorfjellet":        (1606, 1625),
    "origny_large":         (1765, 1813),
    "origny":               (1781, 1813),
    "olifants":             (1880, 1889),
    "groloy":               (1922, 1934),
    "marble_mountain":      (1965, 1979),
    "hippo_pools":          (1947, 1955),
}
SITE = "rockytop"         # difficult case (winter sun, low cross-station match rate)
# SITE = "threeforks"         # easy case, flat ground
# SITE = "butler_landing"
# SITE = "seitah_north"
# SITE = "sid"
# SITE = "whale_mountain"
# SITE = "van_zyl"
# SITE = "olifants"
# SITE = "origny"
# SITE = "groloy"
# SITE = "overlook_mountain"
# SITE = "pico_turquino"
# SITE = "bell_island"
# SITE = "taylorfjellet"
# SITE = "threeforks_south"
# SITE = "rockytop"

ADD_NEARBY_WAYPOINTS = 10                    # also take every waypoint station within this many metres (inclusive)
                                             # of a station imaged in the sol range, any sol; 0 = off

# --- processing ---------------------------------------------------------------------------------------------
REPROCESS_ALL          = 0   # True: process every image again (new masks, radiometry, poses); False: reuse
KEEP_ONLY_REMAINING    = 1   # True: only the images still in processed/images_png8 (delete unsuitable ones
                                 # there); the filters below make this rarely necessary
SKY_ELEVATION_DEG      = 20.0    # Navcam frames pointing higher than this (sky surveys) are not processed
LMST_WINDOW_H          = (8.0, 17.0)   # images taken outside this local mean solar time window are not processed
MAX_SATURATED_FRACTION = 0.05    # images with more than this fraction of their valid pixels saturated are not processed
NAVCAM_BRIGHTNESS      = 0.9     # brightness factor for Navcam (v0p30: 10 % darker to keep nominal exposures in range)
NO_MASK_INFERENCE_AT = [         # stations where the mask model is off (the rover stays in the images)
                            # "S003D0000",    # OEB landing site
                            # "S026D1004",    # Rockytop
                            # "S032D1184",    # Three Forks
                            # "S047D0000",    # Airey Hill
                        ]
STORE_MASK_IN_ALPHA    = True    # also put the mask in the PNG alpha channel (masks/ is written either way)

# --- cameras ------------------------------------------------------------------------------------------------
NAVCAM_DISTORTION = "fisheye_tangential"   # v0p35: "fisheye_tangential" (THIN_PRISM_FISHEYE, sx1 = sy1 = 0: the joint
                                   # calibration's choice, needs NAVCAM_CAMERAS), "rational" (valid to the frame corners;
                                   # NAVCAM_CAMERAS = ...\navcal_v0p40\navcam_joint_rational) or "polynomial" (Metashape K1-K3)
NAVCAM_RIG        = "consensus"    # rig rotation start: "consensus" (refined, v0p22) or "cahv"; baseline = CAHV
NAVCAM_CAMERAS    = None           # folder of verified consensus Navcam cameras from notebook 04 (None: shipped)
NAVCAM_CAMERAS    = Path(r"D:\scapes\colmap\camera_analysis\navcal_v0p40\navcam_joint")   # v0p40: joint calibration of 23 blocks at -20 degC, rig drift from 22 blocks / 25 sol epochs (notebook 04, 2e-2f)
NAVCAM_INTRINSICS = "auto"       # "refine" (default: always refine the Navcam intrinsics), "hold" (keep the start cameras), or "auto" (hold where the network is weak: < 4 stations or < 5 m span)
NAVCAM_RIG_REFINE = "refine"         # "refine" (= "rotation": always refine the rig rotation), "auto" (v0p35: refine it
                                   # on a strong network, hold it on a weak one) or "hold"; the baseline is always held.
                                   # A held rig starts at the joint rig turned to the block's camera temperature and sol
TANGENTIAL        = "refine"       # Navcam p1, p2: "refine", "xml" (hold at the calibration) or "zero"
ZCAM_TANGENTIAL   = "zero"         # v0p40: Mastcam-Z p1, p2: "refine", "xml" (hold at the start camera), "zero"
                                   # (set to 0 and held), or None (as TANGENTIAL); independent of the Navcam setting
INCLUDE_ZCAM34    = False          # add the Mastcam-Z 34 mm frames (ZL0/ZR0 "_034") of the same sols
ZCAM_INTRINSICS   = "focus_model"  # Mastcam-Z start: "focus_model" (v0p22), "label" or "xml"
ZCAM_FOCUS_BIN    = 30             # focus motor counts per Mastcam-Z camera (None: one camera per eye)
ZCAM_BIN_REFINE   = "focal"        # a focus-bin camera refines fx, fy only ("all": also principal point, distortion)
ZCAM_HOLD_F_IMAGES = 2             # focus bins with <= this many images hold f at the focus model
ZCAM_BACKLASH     = "split"        # v0p40: focus backlash states. "split": after the alignment each focus group (eye,
                                   # sol, sequence, focus count) is classified from its images' own focal lengths - the
                                   # dominant backlash state (f ~1 % above CAHVOR, the focus model) or the regular state
                                   # (f ~ CAHVOR, mostly single frames and small mosaics) - and the regular images get
                                   # their own cameras (start f = label; held for <= ZCAM_HOLD_F_IMAGES images) before
                                   # a last adjustment; "report": classify only; None: off
ZCAM_RIG          = True           # True: the two Mastcam-Z eyes of an exposure form a stereo rig; False: each
                                   # image is its own frame

# --- features and matching ----------------------------------------------------------------------------------
MAX_NUM_FEATURES = 16000
SIFT = dict(domain_size_pooling=False, estimate_affine_shape=False)   # DSP-SIFT / affine shape (CPU only, slower)
MATCH_MODE = "exhaustive"          # "exhaustive" (baseline) or "prior_pairs" (quick / CPU)
MATCH = dict(max_ratio=0.9, max_distance=1.0, cross_check=True, guided_matching=False, max_error_px=6.0)
MATCH_BLOCK_SIZE = 100             # images per exhaustive-matching block (COLMAP default 50; larger = fewer reloads)

# --- reconstruction -----------------------------------------------------------------------------------------
# rounds of (triangulation threshold [full-res px], Cauchy scale [sigma], max residual kept [native px])
SCHEDULE = ((24.0, 10.0, 8.0), (12.0, 4.0, 4.0), (8.0, 2.0, 2.0))
MIN_TRI_ANGLE_DEG = 0.25           # keeps stereo-only points out to ~97 m at the 0.42 m baseline
TRIANGULATION = dict(max_transitivity=1)     # COLMAP track building (see mppp.sfm.reconstruction.triangulate)
ATTITUDE_PRIOR_DEG = 2.0           # CAHV attitude prior per frame (1 deg holds the ENU frame more tightly)
LOCALIZE_MIN_IMAGES = 3            # stations (site, drive) with fewer images get no waypoint position prior
EXCLUDE_OUTLIERS = True            # drop frames that do not align with the majority, then re-adjust
OUTLIER_THRESHOLDS = {}            # e.g. {"residual_factor": 3.0, "min_shift_m": 0.25} (reconstruction.OUTLIER_DEFAULTS)

STAGED            = True           # Navcam block first, then Mastcam-Z with the Navcam cameras and rig held
HOLD_CAMERAS      = ()             # further camera keys to hold at their start values, e.g. ("ZL034_f0",)
LINEAR_SOLVER     = "auto"         # bundle adjustment: "auto" (dense Schur up to 600 frames, else iterative), or
                                   # "sparse_schur" / "dense_schur" / "iterative_schur"
THERMAL_BINS_DEG  = 10             # v0p31: Navcam camera-temperature bins for the final stage (one focal length per
                                   # eye and bin; held at the thermal model where the intrinsics are held); None: off
THERMAL_MIN_IMAGES = 8             # a temperature bin with fewer images is merged into its neighbour

In [ ]:
# derived settings
from mppp.sfm.sites import check_sites
for _w in check_sites(SITES):
    print("WARNING: SITES", _w)
SOLS = SITES[SITE]
if SOLS[1] < SOLS[0]:
    raise ValueError(f"SITES[{SITE!r}] = {SOLS}: the end sol is before the start sol")
RIG_REFINE = {"refine": "rotation", "rotation": "rotation", "auto": "auto", "hold": False}.get(NAVCAM_RIG_REFINE)
if RIG_REFINE is None:
    raise ValueError(f"NAVCAM_RIG_REFINE = {NAVCAM_RIG_REFINE!r}: use 'refine', 'auto' or 'hold'")
WORK = SCAPES_ROOT / ((f"{SITE}_colmap_nav_zcam34" if INCLUDE_ZCAM34 else f"{SITE}_colmap"))
for _k, _v in (("TANGENTIAL", TANGENTIAL), ("ZCAM_TANGENTIAL", ZCAM_TANGENTIAL)):
    if _v not in ("refine", "xml", "zero") and not (_k == "ZCAM_TANGENTIAL" and _v is None):
        raise ValueError(f"{_k} = {_v!r}: use 'refine', 'xml' or 'zero'")
_ZT = ZCAM_TANGENTIAL or TANGENTIAL
ZERO_TERMS = ("p1", "p2", "b1", "b2") if TANGENTIAL == "zero" else ("b1", "b2")              # Navcam
ZCAM_ZERO_TERMS = None if _ZT == TANGENTIAL else (("p1", "p2", "b1", "b2") if _ZT == "zero" else ("b1", "b2"))
# p1, p2 refined per camera family (v0p40): a bool when both follow TANGENTIAL
REFINE_TANGENTIAL = ((TANGENTIAL == "refine") if _ZT == TANGENTIAL
                     else {"N": TANGENTIAL == "refine", "Z": _ZT == "refine"})
if GPU_PY:
    print("GPU env:", check_gpu_python(GPU_PY))
print(SITE, SOLS, "->", WORK, "| Mastcam-Z 34 mm:", bool(INCLUDE_ZCAM34), "| reprocess all:", bool(REPROCESS_ALL))
COLMAP_DIR = WORK / "colmap"
# v0p35.1: a WORK folder left by another sol range (e.g. a site renamed in SITES) would mix two blocks
if (COLMAP_DIR / "project.json").is_file():
    _sols = [int(r["sol"]) for r in json.loads((COLMAP_DIR / "project.json").read_text()).get("images", []) if "sol" in r]
    _in = sum(SOLS[0] <= s <= SOLS[1] for s in _sols)
    if _sols and _in == 0:
        print(f"WARNING: {COLMAP_DIR} holds a block of sols {min(_sols)}-{max(_sols)}, none in {SOLS[0]}-{SOLS[1]}: "
              f"rename or move that folder before running {SITE!r} with this sol range")


def banner():                          # first line of every results cell: the site and its COLMAP folder
    print(f"== {SITE}  (sols {SOLS[0]}-{SOLS[1]})  |  {COLMAP_DIR}")

## 1–2. Select and process

In [ ]:
banner()
if ADD_NEARBY_WAYPOINTS:              # the sol range plus other visits within ADD_NEARBY_WAYPOINTS metres of its stations
    from mppp.select import find_imgs_near
    WAYPOINTS = mppp.load_waypoints()
    paths, near = find_imgs_near(PDS_DIR, ["NLF", "NRF"], SOLS, WAYPOINTS, radius_m=ADD_NEARBY_WAYPOINTS)
    print(f"Navcam: {near['n_in_range']} in sols {SOLS[0]}-{SOLS[1]} + {near['n_added']} from other visits within "
          f"{ADD_NEARBY_WAYPOINTS} m")
    for a in near["stations_added"]:
        print(f"   {a['station']} ({a['distance_m']} m from {a['nearest']}): {a['images']} images, sols {a['sols'][0]}-{a['sols'][1]}")
else:
    paths = find_imgs(PDS_DIR, ["NLF", "NRF"], sol_range=SOLS)
prefixes = ("NCAM",)
if INCLUDE_ZCAM34:
    if ADD_NEARBY_WAYPOINTS:
        zp, znear = find_imgs_near(PDS_DIR, ["ZL0", "ZR0"], SOLS, WAYPOINTS, radius_m=ADD_NEARBY_WAYPOINTS, sequ_id="_034")
        print(f"Mastcam-Z 34: {znear['n_in_range']} in range + {znear['n_added']} from other visits")
        paths += zp
    else:
        paths += find_imgs(PDS_DIR, ["ZL0", "ZR0"], sol_range=SOLS, sequ_id="_034")
    prefixes += ("ZCAM",)
paths, sel = select_best_products(paths, sequence_prefix=prefixes)      # one product per exposure
print(json.dumps(sel, indent=1)); print(len(paths), "products")

In [ ]:
banner()
cfg = mppp.default_config()
cfg["export"].update(formats=["PNG8"], store_mask_in_alpha=bool(STORE_MASK_IN_ALPHA), write_mask_files=True,
                     write_colmap=False)
# mask model: the released default ("mppp_mask_v2"); or cfg["masking"]["checkpoint"] = "<path to your checkpoint>"
cfg["masking"]["skip_inference_at"] = NO_MASK_INFERENCE_AT
cfg["selection"].update(max_boresight_elevation_deg=SKY_ELEVATION_DEG,       # sky-pointing Navcam frames
                        lmst_window_h=list(LMST_WINDOW_H) if LMST_WINDOW_H else None,
                        max_saturated_fraction=MAX_SATURATED_FRACTION)
cfg["color"]["brightness_by_family"] = {"N": NAVCAM_BRIGHTNESS}
PROCESSED = WORK / "processed"
# KEEP_ONLY_REMAINING = False: every selected product (images deleted from processed/images_png8 come back).
# REPROCESS_ALL = False: images processed with the same configuration are reused from the manifest.
manifest = mppp.process_images(paths, PROCESSED, cfg, mppp.load_waypoints(), provenance=sel,
                               only_existing="PNG8" if KEEP_ONLY_REMAINING else None,
                               reuse_existing=not REPROCESS_ALL, workers=WORKERS)
print(manifest["n_processed"], "images in the manifest,", len(manifest["failed"]), "failed,",
      len(manifest.get("skipped", [])), "skipped by the selection rules (sky, LMST window, saturation)")
skipped = [m["source_product"] for m in manifest["images"] if m["mask"].get("inference_skipped")]
if skipped:
    print(len(skipped), "images without mask inference (rover visible):",
          sorted({f"S{m['site']:03d}D{m['drive']:04d}" for m in manifest["images"] if m["mask"].get("inference_skipped")}))
if manifest.get("only_existing"):
    print("kept", manifest["only_existing"]["n_kept"], "of", manifest["only_existing"]["n_selected"], "selected;",
          manifest["only_existing"]["n_removed"], "removed")

## 3–4. Project, features, database

In [ ]:
banner()
from collections import Counter
import numpy as np
PROJECT = WORK / "colmap"
from mppp.sfm.project import navcam_cameras_fingerprint
proj = SfmProject.load(PROJECT) if (PROJECT / "project.json").is_file() else None
wanted = {"zero_terms": list(ZERO_TERMS), "zcam_intrinsics": ZCAM_INTRINSICS,
          "zcam_zero_terms": list(ZCAM_ZERO_TERMS) if (ZCAM_ZERO_TERMS is not None and INCLUDE_ZCAM34) else None,
          "zcam_focus_bin": float(ZCAM_FOCUS_BIN) if ZCAM_FOCUS_BIN else None, "zcam_bin_refine": ZCAM_BIN_REFINE,
          "zcam_rig": ZCAM_RIG, "navcam_distortion": NAVCAM_DISTORTION, "navcam_rig": NAVCAM_RIG,
          "zcam_hold_f_images": int(ZCAM_HOLD_F_IMAGES) if ZCAM_INTRINSICS == "focus_model" else None,
          "prior_R_corrected": True}
names = {Path(m["source_product"]).stem for m in manifest["images"]}
if (REPROCESS_ALL or proj is None or names != {r["stem"] for r in proj.images}    # other images than the manifest
        or any(proj.settings.get(k) != v for k, v in wanted.items())            # or different camera settings
        or (proj.settings.get("navcam_cameras") or {}).get("dir") != (str(NAVCAM_CAMERAS) if NAVCAM_CAMERAS else None)
        or (NAVCAM_CAMERAS and (proj.settings.get("navcam_cameras") or {}).get("fingerprint")
            != navcam_cameras_fingerprint(NAVCAM_CAMERAS))):     # v0p31: a rewritten consensus rebuilds the project
    proj = SfmProject.create(manifest["images"], PROCESSED, PROJECT, prior_sigma_m=(1.0, 1.0, 1.0),
                             zcam_intrinsics=ZCAM_INTRINSICS, zero_terms=ZERO_TERMS,
                             zcam_focus_bin=ZCAM_FOCUS_BIN, zcam_bin_refine=ZCAM_BIN_REFINE, zcam_rig=ZCAM_RIG,
                             navcam_distortion=NAVCAM_DISTORTION, navcam_rig=NAVCAM_RIG,
                             zcam_hold_f_images=ZCAM_HOLD_F_IMAGES,
                             navcam_cameras=NAVCAM_CAMERAS,
                             zcam_zero_terms=ZCAM_ZERO_TERMS if INCLUDE_ZCAM34 else None)
else:
    proj.refresh_images(manifest["images"])      # copies (no hard links) follow images processed again
if COLMAP_BAT:
    proj.settings["colmap_bat"] = COLMAP_BAT     # open_in_colmap.bat starts this COLMAP
    proj.save()
print(len(proj.images), "images;", {k: round(v["baseline_m"], 4) for k, v in proj.rig.items()}, "m baseline;",
      {k: v["n_pairs"] for k, v in proj.rig.items()}, "CAHV pairs;",
      {k: v.get("rotation_source", "CAHV") for k, v in proj.rig.items()}, "rig rotation")
for k, c in proj.cameras.items():
    if not c.get("group"):
        print(k, [round(x, 3) for x in c["params"][:5]], f"p1 {c['params'][6]:.2e} p2 {c['params'][7]:.2e}",
              f"k3 {c['params'][8]:.3g}", f"k4 {c['params'][9]:.3g} (rational)" if c.get("free_params") else "")
bins = Counter(c["group"] for c in proj.cameras.values() if c.get("group"))
for g, n in sorted(bins.items()):
    cs = [c for c in proj.cameras.values() if c.get("group") == g]
    held = sum("fx" in (c.get("fixed_params") or []) for c in cs)
    print(f"{g}: {n} focus bins ({held} with f held), counts {min(c['focus_count_range'][0] for c in cs):.0f}-"
          f"{max(c['focus_count_range'][1] for c in cs):.0f}, images per bin {sorted(c['n_images'] for c in cs)}")
corr = [r["prior_R_correction_deg"] for r in proj.images if "prior_R_correction_deg" in r]
if corr:   # Mastcam-Z label attitudes rotated to fit the camera's principal point (v0p14.5)
    print(f"Mastcam-Z prior attitude correction: median {np.median(corr):.3f} deg, max {max(corr):.3f} deg ({len(corr)} images)")
print("images per camera:", dict(Counter(r["instrument"] for r in proj.images)))

In [ ]:
banner()
# SIFT at native resolution with masks -> features.db.  Reused on reruns only if features.json beside it records
# the same settings and image/mask files; overwrite=True always redoes it.
extract_features(proj, max_num_features=MAX_NUM_FEATURES, use_gpu=True if GPU_PY else None, python=GPU_PY,
                 overwrite=bool(REPROCESS_ALL), **SIFT)
print(build_database(proj))          # replaces database.db (and any matches in it) on every run

## 5. Matching

In [ ]:
banner()
if MATCH_MODE == "prior_pairs":
    pairs = prior_overlap_pairs(proj, min_overlap=0.05)
    print(len(pairs), "prior pairs")
print(match(proj, mode=MATCH_MODE, use_gpu=True if GPU_PY else None, python=GPU_PY, block_size=MATCH_BLOCK_SIZE, **MATCH))

## 6. Reconstruct and refine

In [ ]:
banner()
from mppp.sfm.reconstruction import navcam_network
net = navcam_network(proj)
print(f"Navcam network: {net['stations']} stations over {net['span_m']:.1f} m, {net['navcam_images']} images -> {net['verdict']}"
      f" (intrinsics setting {NAVCAM_INTRINSICS!r})")
from mppp.sfm.thermal import image_temperatures, thermal_model_for_project
TEMPS = image_temperatures(proj, pds_dir=PDS_DIR if THERMAL_BINS_DEG else None, verbose=bool(THERMAL_BINS_DEG))
rec = reconstruct(proj, sigma_px=0.5,           # keypoint sigma in native pixels
                  schedule=SCHEDULE, refine_rig=RIG_REFINE,
                  refine_tangential=REFINE_TANGENTIAL,   # Navcam TANGENTIAL, Mastcam-Z ZCAM_TANGENTIAL
                  min_track_length=2,           # two-view tie points (e.g. seen only by one stereo pair) are kept
                  min_tri_angle_deg=MIN_TRI_ANGLE_DEG, attitude_prior_deg=ATTITUDE_PRIOR_DEG,
                  triangulation_options=TRIANGULATION,
                  exclude_outliers=EXCLUDE_OUTLIERS, outlier_thresholds=OUTLIER_THRESHOLDS,
                  navcam_intrinsics=NAVCAM_INTRINSICS, staged=STAGED, hold_cameras=HOLD_CAMERAS,
                  linear_solver=LINEAR_SOLVER, temperatures=TEMPS, thermal_bins_deg=THERMAL_BINS_DEG,
                  thermal_min_images=THERMAL_MIN_IMAGES, thermal_model=thermal_model_for_project(proj),
                  localize_min_images=LOCALIZE_MIN_IMAGES,
                  zcam_backlash=ZCAM_BACKLASH if INCLUDE_ZCAM34 else None)
rs = proj.settings["reconstruction"]
print("Navcam intrinsics:", rs.get("navcam_intrinsics"), "| held cameras:", rs.get("hold_cameras") or "none",
      "| staged:", rs.get("staged"))
excluded = rs.get("excluded", [])
print(len(excluded), "frames excluded as outliers")
for st, v in (rs.get("unlocalized") or {}).items():
    print(f"   no position prior: {st} ({v['images']} images, {v['cross_points']} cross-station points): {v['prior']}")
for e in excluded:
    print("  ", ", ".join(e["images"]), "|", e["station"], "|", "; ".join(e["reasons"]))
summary = export_for_error(proj, rec)
print(json.dumps({k: summary[k] for k in ("tracks", "residual_rms_native_px", "registered_images", "project_images",
                                           "prior_offset_median_m", "station_components")}, indent=1))
summary_nav = None
if rs.get("navcam_stage"):
    # the stage-1 Navcam-only solution: same images, features and database as the combined one
    import pycolmap
    rec_nav = pycolmap.Reconstruction(str(proj.root / rs["navcam_stage"]["path"]))
    summary_nav = export_for_error(proj, rec_nav, out_dir=proj.root / "error_input_navcam")
    print("Navcam-only stage exported to error_input_navcam/:",
          {k: summary_nav[k] for k in ("tracks", "residual_rms_native_px", "registered_images")})
conv = convergence_statistics(rec, proj)
proj.settings["reconstruction"]["convergence"] = conv
proj.save()
print("tie points by convergence angle", conv["bins_deg"], "deg:", conv["points"],
      "| cross-station:", conv["cross_station_points"], f"| > 10 deg: {conv['points_over_10deg']}, "
      f"> 20 deg: {conv['points_over_20deg']}")
# start -> refined camera parameters (Mastcam-Z focus bins: section 7b).  Cameras are looked up by database id:
# after the thermal stage an eye's camera can have no images left (all frames in temperature bins).
from mppp.sfm.reconstruction import print_camera_changes
print_camera_changes(rec, proj)
print("COLMAP GUI: double-click", proj.root / "open_in_colmap.bat")
_bl = proj.settings.get("zcam_backlash")
if _bl:   # v0p40: Mastcam-Z focus states (per image: colmap/zcam_focus_states.json)
    print("Mastcam-Z focus groups:", _bl["group_counts"], "| f / label f:",
          {s: (round(v["ratio_median"], 4) if v["ratio_median"] else None) for s, v in (_bl.get("after") or _bl["before"]).items()},
          "| regular-state cameras:", [(r["camera"], r["images"], round(r.get("f_refined_px", r["start_f_px"]), 1)) for r in _bl["cameras"]])


## 7. Alignment health — check before any analysis
Each check has provisional warn and fail thresholds. These were set a priori and are not validated limits; pass `thresholds={...}` to change them. The report is written to `<project>/health/` (`health.json`, `health.md`, `health.png`, and `station_map.png`: a top-down view of the stations with the prior-to-refined shift of every camera).

| section | checks |
|---|---|
| registration | images registered; frames held at their prior (< 30 observations) |
| tie points | points; track-length population (two-view share); observations per image; cross-station ties; tied station blocks |
| reprojection | median and 95th-percentile residual (native px); left vs right eye; outlier images; residual growth towards the image edge; tie-point coverage by image radius |
| camera model | change of f, principal point and k1–k4; image displacement of the same ray |
| stereo rig | right-camera rotation vs its start; spread of the CAHV pairs |
| poses | refined − prior centre per station; spread within a station; attitude change; scale vs the priors |

In [ ]:
banner()
if "rec" not in globals():         # v0p40: after a kernel restart, read the alignment section 6 saved
    from mppp.sfm.reconstruction import load_alignment
    rec = load_alignment(proj)       # a clear error if section 6 has not finished for this project
report = assess_alignment(proj, rec)
print(health_table(report))
files = write_health(report, proj.root / "health", rec, proj)
from IPython.display import Image, display
display(Image(filename=files["png"]))
if files.get("station_map"):                       # top-down view: stations, prior -> refined shifts (health/station_map.png)
    display(Image(filename=files["station_map"]))
if report["verdict"] == "fail":
    print("\nFAIL: fix the alignment (matching, priors, masks, thresholds) before any error analysis.")

### 7b. Mastcam-Z focus breathing
Written to `<project>/health/zcam_focus_breathing.{png,csv,json}`. Notebook 04 compares scapes.

In [ ]:
banner()
if "rec" not in globals():         # v0p40: after a kernel restart, read the alignment section 6 saved
    from mppp.sfm.reconstruction import load_alignment
    rec = load_alignment(proj)       # a clear error if section 6 has not finished for this project
from mppp.sfm.zcam import write_focus_breathing
if any(c.get("group") for c in proj.cameras.values()):
    fb = write_focus_breathing(proj, rec)
    for g, f in fb["fits"].items():
        for kind in ("refined", "label"):
            if f[kind]:
                print(f"{g} {kind:7s}: f = {f[kind]['f0_px']:.1f} px at count {f['reference_count']:.0f}, "
                      f"slope {f[kind]['slope_px_per_count']:+.4f} px/count, rms {f[kind]['rms_px']:.2f} px, n = {f[kind]['n']}")
    display(Image(filename=fb["png"]))
else:
    print("no Mastcam-Z focus-bin cameras")

## 8. Export for the error analysis
Notebook 04 (`04_camera_models.ipynb`) compares the camera models of the sites and writes the consensus cameras; notebook 05 (`05_error_analysis.ipynb`) then reads the `error_input/` folders.

In [ ]:
banner()
import csv
import matplotlib.pyplot as plt
from mppp.sfm.export import plot_camera_shifts
pose_rows = list(csv.DictReader(open(proj.root / "error_input" / "poses.csv")))
fig = plot_camera_shifts(proj, rows=pose_rows, out_png=proj.root / "error_input" / "camera_shifts.png")
plt.close(fig)                                     # the same figure is in health/station_map.png (section 7)
if report["verdict"] == "fail":
    print("alignment health is FAIL (section 7): fix the alignment before using error_input/ in notebook 05")
eps = summary.get("eps", {})
print("error_input:", proj.root / "error_input")
print(f"eps (rms residual / sqrt 2): same-station tracks {eps.get('eps_intra_px', float('nan')):.3f} px, "
      f"cross-station tracks {eps.get('eps_cross_px', float('nan')):.3f} px")